In [2]:
import pandas as pd
import random

# A mix of realistic review templates
templates = [
    "I absolutely love this product! It works exactly as described.",
    "Terrible experience. The item broke after just two days of use...",
    "It's decent for the price, but don't expect premium quality.",
    "Delivery was super fast, and the packaging was great. 5 stars!",
    "DO NOT BUY THIS! Total waste of money.",
    "The battery life is amazing, highly recommend it.",
    "Customer service was unhelpful when I tried to return it.",
    "Works well, but the instruction manual is very confusing.",
    "Exactly what I was looking for. Perfect fit!",
    "Item arrived damaged. Very disappointed.",
    "Good quality material, feels very sturdy.",
    "Overpriced for what it actually does.",
    "I bought this as a gift and my friend loves it.",
    "The color is slightly different from the picture, but still nice.",
    "Stopped working after a month. Wouldn't recommend."
]

# Generate 120 reviews to satisfy the "minimum 100" requirement
random.seed(42) # For reproducibility
reviews = []
for _ in range(120):
    review = random.choice(templates)
    # Add some noise (extra punctuation/casing) to make preprocessing meaningful
    if random.random() > 0.7:
        review = review.lower()
    if random.random() > 0.5:
        review += "!!!"
    reviews.append(review)

df = pd.DataFrame({'review_text': reviews})
df.to_csv('product_reviews.csv', index=False)
print("Successfully created product_reviews.csv with 120 reviews!")

Successfully created product_reviews.csv with 120 reviews!


In [3]:
import pandas as pd
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download required NLTK data files (only needs to be run once)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

# Load the dataset
df = pd.read_csv('product_reviews.csv')

# Initialize NLP tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # 1. Convert text to lowercase
    text = str(text).lower()
    
    # 2. Tokenization (splitting text into individual words)
    tokens = word_tokenize(text)
    
    # 3. Remove punctuation
    tokens = [word for word in tokens if word not in string.punctuation]
    
    # 4. Remove stopwords (and, a, the, etc.)
    tokens = [word for word in tokens if word not in stop_words]
    
    # 5. Lemmatization (converting words to their base form, e.g., 'running' -> 'run')
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    # Join tokens back into a single string for feature extraction later
    return ' '.join(tokens)

# Apply the pipeline to our dataset
df['cleaned_review'] = df['review_text'].apply(preprocess_text)

# Let's look at the before and after!
print(df[['review_text', 'cleaned_review']].head())

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\emaideb\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\emaideb\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\emaideb\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\emaideb\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


                                         review_text  \
0       Good quality material, feels very sturdy.!!!   
1  Delivery was super fast, and the packaging was...   
2       good quality material, feels very sturdy.!!!   
3           Item arrived damaged. Very disappointed.   
4  Delivery was super fast, and the packaging was...   

                               cleaned_review  
0          good quality material feel sturdy.  
1  delivery super fast packaging great 5 star  
2          good quality material feel sturdy.  
3           item arrived damaged disappointed  
4  delivery super fast packaging great 5 star  


In [4]:
from collections import Counter
import pandas as pd

# 1. Combine all cleaned reviews into one giant list of words
all_words = ' '.join(df['cleaned_review']).split()

# 2. Count the frequency of every single word
word_counts = Counter(all_words)

# 3. Print Vocabulary Size (the number of unique words)
vocab_size = len(word_counts)
print(f"Total Vocabulary Size: {vocab_size} unique words")
print("-" * 30)

# 4. Analyze Top Frequent Words
top_10_words = word_counts.most_common(10)

print("Top 10 Most Frequent Words:")
# We will format this into a nice table using pandas for better readability
top_words_df = pd.DataFrame(top_10_words, columns=['Word', 'Frequency'])
print(top_words_df.to_string(index=False))

Total Vocabulary Size: 81 unique words
------------------------------
Top 10 Most Frequent Words:
     Word  Frequency
      n't         21
  exactly         19
     love         18
     item         16
recommend         16
  stopped         15
  working         15
    month         15
    would         15
  looking         12


In [5]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Assuming 'df' is your DataFrame from Task 1 and contains the 'cleaned_review' column

# ---------------------------------------------------------
# 1. One Hot Encoding (Document-level vector)
# ---------------------------------------------------------
# binary=True ensures we only track if a word is present (1) or absent (0)
ohe_vectorizer = CountVectorizer(binary=True)
ohe_matrix = ohe_vectorizer.fit_transform(df['cleaned_review'])

print("--- 1. One Hot Encoding ---")
print(f"Matrix Shape: {ohe_matrix.shape}")
# Let's peek at the first review's vector (showing only the first 20 columns for readability)
print(f"Sample Vector (first review): {ohe_matrix.toarray()[0][:20]}\n")


# ---------------------------------------------------------
# 2. Bag of Words (BoW)
# ---------------------------------------------------------
# Default CountVectorizer counts the actual frequency of each word in the document
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(df['cleaned_review'])

print("--- 2. Bag of Words (CountVectorizer) ---")
print(f"Matrix Shape: {bow_matrix.shape}")
print(f"Sample Vector (first review): {bow_matrix.toarray()[0][:20]}\n")


# ---------------------------------------------------------
# 3. TF-IDF (Term Frequency-Inverse Document Frequency)
# ---------------------------------------------------------
# TfidfVectorizer calculates a weighted score. 
# Words that appear often in one review but rarely across all reviews get a higher score.
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(df['cleaned_review'])

print("--- 3. TF-IDF ---")
print(f"Matrix Shape: {tfidf_matrix.shape}")
print(f"Sample Vector (first review): {tfidf_matrix.toarray()[0][:20]}\n")

--- 1. One Hot Encoding ---
Matrix Shape: (120, 69)
Sample Vector (first review): [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

--- 2. Bag of Words (CountVectorizer) ---
Matrix Shape: (120, 69)
Sample Vector (first review): [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]

--- 3. TF-IDF ---
Matrix Shape: (120, 69)
Sample Vector (first review): [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]



In [6]:
import pandas as pd

# Let's create a test sentence with repeated words to see the models behave differently
test_review = ["this product is a good product but the battery is very very bad"]

# 1. Transform this single sentence using the vectorizers we already trained in Task 3
test_ohe = ohe_vectorizer.transform(test_review)
test_bow = bow_vectorizer.transform(test_review)
test_tfidf = tfidf_vectorizer.transform(test_review)

# 2. Get the vocabulary words
feature_names = bow_vectorizer.get_feature_names_out()

# 3. Create a DataFrame to compare the non-zero values side-by-side
comparison_data = []

# We loop through the BoW array to find which words actually appeared (count > 0)
for col_index in test_bow.nonzero()[1]:
    word = feature_names[col_index]
    
    # Extract the score for this specific word from all three models
    ohe_score = test_ohe[0, col_index]
    bow_score = test_bow[0, col_index]
    tfidf_score = test_tfidf[0, col_index]
    
    comparison_data.append([word, ohe_score, bow_score, round(tfidf_score, 4)])

# Display the final comparison
comparison_df = pd.DataFrame(comparison_data, columns=['Word', 'OHE Score', 'BoW Score', 'TF-IDF Score'])
print("\n--- Task 4: Feature Comparison Analysis ---")
print("Test Sentence: 'this product is a good product but the battery is very very bad'")
print("-" * 65)
print(comparison_df.to_string(index=False))


--- Task 4: Feature Comparison Analysis ---
Test Sentence: 'this product is a good product but the battery is very very bad'
-----------------------------------------------------------------
   Word  OHE Score  BoW Score  TF-IDF Score
battery          1          1        0.4082
   good          1          1        0.4082
product          1          2        0.8165
